# SplatAD / OmniRE trên PandaSet

Đọc PandaSet từ Google Drive, train trên Colab GPU và phát lại toàn cảnh qua ba camera trước. Dashboard có các thẻ video **Gốc**, **SplatAD**, **R2**, **R3**, **OmniRe** (render từ checkpoint cuối ở 60 FPS), bảng PSNR/SSIM bốn mô hình và mục **Pose SMPL**.

- Mặc định dùng `STAGE="scene"` cho cảnh `030`: SplatAD gốc, R2, R3 và OmniRe gốc train trên cùng sáu camera, LiDAR Pandar64, 10.000 bước và bộ ảnh đo. Checkpoint đã đủ bước được dùng lại.
- `STAGE="curriculum"` chạy `028` (gần) → `011` (nhiều người) → `008` (xa và bị che). Mỗi cảnh train một mô hình riêng với sáu camera và xuất MP4 60 FPS của ba camera trước. Sửa `SCENES` để chọn cảnh khác.
- Mặc định ưu tiên `MyDrive/SplatAD+OmniRe/pandaset.zip`. Có thể đặt `PANDASET_PATH` tới một ZIP khác hoặc thư mục đã giải nén chứa các cảnh `001`, `011`, …
- R2/R3 chạy quy trình trích xuất của OmniRe trên sáu camera với SMPL Neutral v1.1. Đặt `basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl` hoặc thư mục gói SMPL chính thức đã giải nén vào thư mục dự án trên Drive. Có thể xem video gốc trước bước này.

Drive lưu dữ liệu chuẩn bị trong `inputs/` và `pose-init/`, checkpoint trong `runs/`, video và metric trong `artifacts/`. Colab giải nén tạm các cảnh cần dùng vào đĩa runtime.

Các clip camera gốc được tạo ở cell xem trước từ Drive. Các clip tái tạo được xuất sau khi mỗi mô hình có checkpoint. [PandaSet](https://huggingface.co/datasets/georghess/pandaset) · [SMPL](https://smpl.is.tue.mpg.de/) · [Pose OmniRe](https://github.com/ziyc/drivestudio/blob/e59bda4fa681f829dbb1d65f0de582b0f633c450/docs/HumanPose.md)

In [ ]:
from collections import Counter
from pathlib import Path
import json
import shutil
import subprocess
import sys
import time
from zipfile import is_zipfile

from google.colab import drive, output

DRIVE = Path("/content/drive")
drive.mount(str(DRIVE))
PERSIST = DRIVE / "MyDrive/SplatAD+OmniRe"
PERSIST.mkdir(exist_ok=True)
REPO = Path("/content/SplatAD-OmniRE")
GITHUB = "https://github.com/buide03/SplatAD-OmniRE.git"
BRANCH = "r3-pedestrian"  # nhánh thử nghiệm: SplatAD, R2, R3 và OmniRe trên 6 camera
ARCHIVE = PERSIST / "SplatAD-OmniRE.zip"

progress = None

def shell(*args, cwd=None):
    if progress is not None:
        return progress.run(args, cwd)
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

if (REPO / ".git").is_dir():
    # Runtime cũ có thể đang ở nhánh khác: chuyển sang bản mới nhất của BRANCH.
    shell("git", "fetch", "origin", BRANCH, cwd=REPO)
    shell("git", "checkout", "-B", BRANCH, f"origin/{BRANCH}", cwd=REPO)
elif not (REPO / "versions.json").exists():
    try:
        shell("git", "clone", "--branch", BRANCH, GITHUB, REPO)
    except subprocess.CalledProcessError:
        # Chỉ dùng bản zip trên Drive khi không clone được; bản zip có thể cũ hơn nhánh.
        assert ARCHIVE.is_file(), f"Không clone được {GITHUB} nhánh {BRANCH}."
        REPO.mkdir(exist_ok=True)
        shutil.unpack_archive(ARCHIVE, REPO)
curriculum_path = REPO / "protocol/curriculum.json"
if (REPO / ".git").is_dir():
    commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], cwd=REPO, capture_output=True, text=True).stdout
    print(f"Mã nguồn: nhánh {BRANCH} · {commit.strip()}")
assert curriculum_path.is_file(), "Cần bản mã nguồn mới có lộ trình train theo độ khó."

sys.path.insert(0, str(REPO))
from research.colab import ColabProgress, checkpoint_step, latest_checkpoint, retry_environment
from research.pandaset_input import prepare_source, select_source
from research.prepare import dump_json, front_pose_summary, read_json, sha256
from research.sync_video import FRONT_CAMERAS

PANDASET_PATH = ""  # Để trống để ưu tiên pandaset.zip trong thư mục dự án rồi tự dò Drive
STAGE = "scene"  # scene | curriculum | smoke | pilot | confirm
DATA_PROTOCOL_VERSION = 2
# Sáu camera cho mọi lượt, trừ smoke. SplatAD, R2, R3 và OmniRe dùng cùng danh sách này.
TRAIN_CAMERAS = list(FRONT_CAMERAS) if STAGE == "smoke" else [*FRONT_CAMERAS, "left", "right", "back"]
CAMERAS = TRAIN_CAMERAS
tag_cameras = "front3" if tuple(CAMERAS) == FRONT_CAMERAS else "surround"
RUN_ID = f"{STAGE}-{tag_cameras}-fullres-rgbq-v{DATA_PROTOCOL_VERSION}"
VARIANTS = ("R2", "R3")  # SplatAD gốc luôn train trước làm mốc đối chứng
STEPS = {"curriculum": 10_000, "scene": 10_000, "smoke": 200, "pilot": 1200, "confirm": 4000}[STAGE]
# R3: người xa đi bằng SMPL với dáng đi chuẩn của cảnh, keypoint 2D, LiDAR và mặt đáy hộp. Chạy thử
# 10.000 bước như các mô hình khác. Mọi mô hình dùng chung lịch learning rate 30.001 bước, nên đặt
# R3_STEPS = 30_000 về sau sẽ train tiếp từ checkpoint 10.000. Thư mục riêng "R3-gait" để không nạp
# nhầm checkpoint của lượt R3 dùng MLP biến dạng trước đó.
R3_STEPS = STEPS
# Dáng đi chuẩn của từng cảnh có người gần; cảnh không có người gần mượn từ đây.
GAIT_LIBRARY = PERSIST / "gait-library"

def steps_for(variant):
    return R3_STEPS if variant == "R3" else STEPS

def variant_dir(variant):
    return "R3-gait" if variant == "R3" else variant
QUALITY_EVERY = 50 if STAGE == "smoke" else 250
# Mỗi checkpoint khoảng 1,7 GB và bản bị thay thế vào Thùng rác Drive, nên lưu thưa: 10 lần cho 10.000 bước.
CHECKPOINT_EVERY = 1000
CURRICULUM = {entry["scene_id"]: entry for entry in json.loads(curriculum_path.read_text(encoding="utf-8"))["scenes"]}
PILOT = json.loads((REPO / "protocol/pilot.json").read_text(encoding="utf-8"))
if STAGE == "curriculum":
    SCENES = list(CURRICULUM)
elif STAGE == "scene":
    SCENES = ["030"]
elif STAGE == "smoke":
    SCENES = ["011"]
else:
    SCENES = [entry["scene_id"] for entry in PILOT["scenes"]]
REPLAY_CAMERAS = FRONT_CAMERAS
REPLAY_FPS = 60
REPLAY_INTERPOLATION = "smooth"  # smooth | linear
REPLAY_SCALE = 1.0  # Giữ độ phân giải gốc cho từng camera
OMNIRE_REFERENCE = "off"  # off | compare | fill
POSE_PIPELINE = "omnire-option2"
NEED_POSE = any(variant in ("R2", "R3") for variant in VARIANTS)
SMPL_FILENAME = "basicmodel_neutral_lbs_10_207_0_v1.1.0.pkl"
SMPL_PACKAGE = PERSIST / "SMPL_python_v.1.1.0"
package_model = Path("smpl/models") / SMPL_FILENAME
model_paths = (
    PERSIST / SMPL_FILENAME,
    SMPL_PACKAGE / package_model,
    SMPL_PACKAGE / SMPL_PACKAGE.name / package_model,
    PERSIST / "SMPL_NEUTRAL.pkl",
)
SMPL_PICKLE = next((path for path in model_paths if path.is_file()), model_paths[0])
MODEL_VERSION = "smpl_neutral_v1.1"
SMPL_SOURCE_ID = sha256(SMPL_PICKLE)[:12] if NEED_POSE and SMPL_PICKLE.is_file() else None
if NEED_POSE:
    RUN_ID = f"{RUN_ID}-smpl-opacity-0.99-{POSE_PIPELINE}-{OMNIRE_REFERENCE}-{MODEL_VERSION}-{SMPL_SOURCE_ID or 'pending'}"
POSE_OUTPUT = PERSIST / "pose-init" / POSE_PIPELINE / f"{MODEL_VERSION}-{SMPL_SOURCE_ID or 'pending'}"
LOCAL_POSES = POSE_OUTPUT / "local-4d-humans"
REFERENCE_POSES = POSE_OUTPUT / "omnire-reference"
INPUTS = PERSIST / "inputs"
RUNS = PERSIST / "runs"
ARTIFACTS = PERSIST / "artifacts"
RESULTS = ARTIFACTS / RUN_ID
assert OMNIRE_REFERENCE in ("off", "compare", "fill"), "Chọn off, compare hoặc fill."
for folder in (INPUTS, RUNS, ARTIFACTS, RESULTS):
    folder.mkdir(exist_ok=True)
progress = ColabProgress(RESULTS / "progress")

read_cameras = (*FRONT_CAMERAS, "left", "right", "back") if NEED_POSE else tuple(CAMERAS)
project_archive = PERSIST / "pandaset.zip"
dataset_path = PANDASET_PATH or (project_archive if project_archive.is_file() else None)
SOURCE, SCENES = select_source(dataset_path, [DRIVE / "MyDrive", DRIVE / "Shareddrives"], SCENES)

def require_smpl_model():
    if NEED_POSE and not SMPL_SOURCE_ID:
        raise FileNotFoundError(f"Cần SMPL Neutral v1.1. Đặt {SMPL_FILENAME} hoặc thư mục "
                                f"{SMPL_PACKAGE.name} đã giải nén trong {PERSIST} rồi chạy lại cell cấu hình.")

def raw_for(scene):
    with progress.phase(f"stage-{scene}"):
        return prepare_source(SOURCE, Path("/content/pandaset-from-drive"), [scene], read_cameras)

def publish_source(scene, raw=None, manifest=None):
    folder = ARTIFACTS / "source" / scene
    path = folder / "run.json"
    if not path.is_file() or not (folder / "front-triptych.mp4").is_file():
        shell(sys.executable, "-m", "research.source_preview", "--raw", raw or raw_for(scene),
              "--scene", scene, "--front-triptych", "--output", folder, cwd=REPO)
    if manifest:
        record = read_json(path)
        summary = front_pose_summary(read_json(manifest))
        if summary != record.get("pose_summary"):
            record["pose_summary"] = summary
            dump_json(path, record)

print(f"PandaSet: {SOURCE}")
print(f"Tiến độ và log trên Drive · {progress.folder}")
if SMPL_SOURCE_ID:
    print(f"SMPL Neutral · {MODEL_VERSION} · {SMPL_PICKLE.name}")
print("Thứ tự train", " → ".join(SCENES))

## Xem video Gốc và Pose SMPL

Cell này tạo video **Gốc** đồng bộ ba camera trước của cảnh đầu tiên và mở dashboard. Video ở thẻ **Render** xuất hiện sau khi có checkpoint. Mục **Pose SMPL** tự cập nhật khi chạy R2 hoặc R3.

In [ ]:
if shutil.which("ffmpeg") is None:
    shell("apt-get", "-qq", "update")
    shell("apt-get", "-qq", "install", "-y", "ffmpeg")

publish_source(SCENES[0])

if "dashboard" in globals() and dashboard.poll() is None:
    dashboard.terminate()
    dashboard.wait(timeout=5)
dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                              "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
time.sleep(1)
assert dashboard.poll() is None, "Không khởi động được dashboard."
output.serve_kernel_port_as_iframe(8765, height=1000)

## Môi trường train

Chọn GPU trong Colab trước khi chạy cell này. Sau khi runtime bị ngắt, kết nối lại và chạy lại notebook với cùng cấu hình lượt train. Bootstrap dùng Python 3.12, PyTorch CUDA 12.8 hoặc 12.6 và fork GSplat theo commit đã pin. Sau khi nạp checkpoint, repo render ba camera theo cùng mốc thời gian rồi ghép thành một clip.

In [ ]:
def bootstrap(profile, repair=False):
    options = ("--repair",) if repair else ()
    phase = f"repair-{profile}" if repair else f"environment-{profile}"
    with progress.phase(phase):
        shell("bash", "scripts/bootstrap-colab.sh", profile, *options, cwd=REPO)

def research(module, *args):
    def launch():
        shell("bash", "-c", 'source "$1"; shift; exec python -m "$@"',
              "splatad", REPO / ".deps/train-env/activate-research", f"research.{module}", *args, cwd=REPO)

    if module == "train":
        return launch()
    return retry_environment(launch, lambda: bootstrap("train", repair=True))

with progress.phase("check-runtime"):
    require_smpl_model()
    shell("nvidia-smi")
bootstrap("train")

## Pose khởi tạo và manifest

OmniRe Option 2 chiếu cuboid người lên ảnh, chạy 4D-Humans trên sáu camera rồi chạy Tracklet Matching và Pose Completion. Mặc định `OMNIRE_REFERENCE="off"` dùng track do quy trình này tạo. Chọn `"compare"` để đo chênh lệch với pose công bố, hoặc `"fill"` cho lần chạy thử kết hợp hai nguồn. Kết quả lưu trong `MyDrive/SplatAD+OmniRe/pose-init/omnire-option2/`.

In [ ]:
BODY = POSE_OUTPUT / "neutral.npz"

def convert_body():
    shell("bash", "-c", 'source "$1"; shift; exec python -m research.prepare "$@"', "splatad",
          REPO / ".deps/smpl-convert/activate-research", "prepare-smpl",
          "--input", SMPL_PICKLE, "--output", BODY, "--model-version", MODEL_VERSION, cwd=REPO)

if NEED_POSE:
    with progress.phase("prepare-smpl-model"):
        require_smpl_model()
        if not is_zipfile(BODY):
            bootstrap("convert")
            retry_environment(convert_body, lambda: bootstrap("convert", repair=True))

pose_env_ready = False

def pose_folder(scene):
    return POSE_OUTPUT / "scenes" / scene / f"export-v{DATA_PROTOCOL_VERSION}"

def tracks_for(scene):
    return pose_folder(scene) / OMNIRE_REFERENCE / "tracks.json"

def manifest_for(scene):
    if NEED_POSE:
        return tracks_for(scene).with_name(f"manifest-{tag_cameras}.json")
    return INPUTS / f"{scene}-{tag_cameras}-rigid-manifest-v{DATA_PROTOCOL_VERSION}.json"

def prepare_pose(scene, raw):
    global pose_env_ready
    final = tracks_for(scene)
    report = final.with_name("comparison.json")
    if final.is_file() and report.is_file():
        return
    primary = pose_folder(scene) / "tracks-4d-humans.json"
    if not primary.is_file():
        local = LOCAL_POSES / scene
        if not all((local / file).is_file() for file in ("humanpose/smpl.pkl", "instances/instances_info.json")):
            if not pose_env_ready:
                bootstrap("pose")
                pose_env_ready = True
            retry_environment(
                lambda: shell("bash", "scripts/omnire-pose.sh", raw, LOCAL_POSES, scene, SMPL_PICKLE, cwd=REPO),
                lambda: bootstrap("pose", repair=True),
            )
        research("prepare", "export-tracks", "--raw", raw, "--processed", LOCAL_POSES,
                 "--body", BODY, "--scene", scene, "--output", primary)
    reference_args = []
    if OMNIRE_REFERENCE != "off":
        reference = pose_folder(scene) / "tracks-omnire-reference.json"
        source_pose = REFERENCE_POSES / scene / "humanpose/smpl.pkl"
        if not reference.is_file():
            if not source_pose.is_file():
                research("omnire", "--raw", raw, "--processed", REFERENCE_POSES, "--allow-missing",
                         "--archive", INPUTS / "pandaset_processed_humanpose.zip", "--scenes", scene)
            if source_pose.is_file():
                research("prepare", "export-tracks", "--raw", raw, "--processed", REFERENCE_POSES,
                         "--body", BODY, "--scene", scene, "--output", reference)
        if reference.is_file():
            reference_args = ["--reference", reference]
    research("pose_init", "--primary", primary, *reference_args, "--body", BODY,
             "--mode", OMNIRE_REFERENCE, "--output", final, "--report", report)
    print(f"Pose đã lưu trên Drive · {final.parent}")

def check_curriculum(scene, spec):
    stage = CURRICULUM[scene]
    selected = set(spec["selected_uuids"])
    moving = [row for row in spec["observations"]
              if row["uuid"] in selected and row["smpl_valid"] and not row["stationary"]]
    near = {row["uuid"] for row in moving if row["bin"] == "near" and row["usable"]}
    per_frame = Counter((row["camera"], row["frame"]) for row in moving if row["usable"])
    simultaneous = max(per_frame.values(), default=0)
    far_occluded = {row["uuid"] for row in moving if row["bin"] == "far"
                    and row["limbs_visible"] is False and row["pose_monitor"]}
    print(f"{scene} · {stage['label']} · {len(near)} track gần · đỉnh {simultaneous} người cùng khung · "
          f"{len(far_occluded)} track xa có mốc bị che để monitor")
    qualifies = {"near": bool(near), "multiple": simultaneous >= 3, "far_occluded": bool(far_occluded)}
    if not qualifies[stage["focus"]]:
        raise ValueError(f"Manifest {scene} thiếu quan sát SMPL cho trọng tâm {stage['label']}")

def prepare_scene(scene, raw):
    tracks, manifest = tracks_for(scene), manifest_for(scene)
    if NEED_POSE:
        prepare_pose(scene, raw)
    if not manifest.exists():
        review = INPUTS / f"{scene}-review.json"
        if not review.is_file():
            review = REPO / "protocol" / f"{scene}-review.json"
        research("prepare", "inventory", "--raw", raw, "--scene", scene,
                 "--cameras", *CAMERAS, "--output", manifest,
                 *(["--tracks", tracks] if NEED_POSE else []),
                 *(["--review", review] if review.is_file() else []))
    if NEED_POSE:
        spec = json.loads(manifest.read_text(encoding="utf-8"))
        print(scene, len(spec["selected_uuids"]), "UUID định tuyến SMPL")
        assert spec["selected_uuids"], f"Cần track SMPL hợp lệ cho cảnh {scene}"
        if STAGE == "curriculum" and scene in CURRICULUM:
            check_curriculum(scene, spec)
        if STAGE in ("pilot", "confirm"):
            simultaneous = {}
            for row in spec["observations"]:
                if row["usable"] and row["smpl_valid"] and row["uuid"] in spec["selected_uuids"]:
                    simultaneous.setdefault((row["camera"], row["frame"]), set()).add(row["uuid"])
            assert max(map(len, simultaneous.values()), default=0) >= 2, f"Cảnh {scene} cần hai UUID SMPL cùng xuất hiện"

## Train và xuất toàn cảnh theo từng checkpoint

Mỗi cảnh chạy SplatAD gốc trước, rồi train các cấu hình nghiên cứu trên cùng bộ ảnh và crop cố định. Tất cả các lượt train RGB ở độ phân giải gốc từ bước 0. Dashboard cập nhật PSNR và SSIM của toàn cảnh, xe và người từ checkpoint đầu, theo mốc và khi hoàn tất. Với R2 và R3, mỗi 250 bước notebook chiếu mesh SMPL khởi tạo và mesh đang tinh chỉnh của từng người đi bộ lên RGB gốc, và ghi độ dịch chuyển tịnh tiến, xoay gốc, xoay khớp so với khởi tạo. Notebook lưu checkpoint mỗi 100 bước và tiếp tục từ checkpoint hợp lệ mới nhất của từng cảnh. Log và trạng thái từng giai đoạn nằm trong thư mục progress trên Drive. R2 và R3 dùng chung mã run với SplatAD để so ở cùng bước. Trước lượt R3 đầu tiên, notebook fit mặt đường cục bộ từ LiDAR quanh chân người ở gần và lưu cạnh manifest.

In [ ]:
def run_folder(scene, variant):
    return RUNS / f"{scene}-{variant_dir(variant)}" / "splatad-smpl" / RUN_ID

def checkpoint_for(scene, variant):
    return latest_checkpoint(run_folder(scene, variant))

def replay_args(scene, variant, zero_velocity=False):
    checkpoint = checkpoint_for(scene, variant)
    suffix = "-velocity-off" if zero_velocity else ""
    folder = RESULTS / f"{scene}-{variant_dir(variant)}{suffix}-{checkpoint.stem}-{REPLAY_FPS}fps-{REPLAY_SCALE:g}x-{REPLAY_INTERPOLATION}"
    shared = ["--config", run_folder(scene, variant) / "config.yml", "--checkpoint", checkpoint,
              "--manifest", manifest_for(scene), "--output", folder]
    if zero_velocity:
        shared.append("--zero-smpl-velocity")
    return folder, shared

def replay_ready(folder):
    artifact = folder / "run.json"
    if not artifact.is_file():
        return False
    record = read_json(artifact)
    triptych = record.get("front_triptych") or {}
    files = [entry["file"] for entry in record.get("videos", [])]
    return (triptych.get("cameras") == list(REPLAY_CAMERAS) and len(files) == len(REPLAY_CAMERAS)
            and all((folder / file).is_file() and (folder / file).stat().st_size
                    for file in [*files, triptych["file"]]))

def render_scene(scene, variant, zero_velocity=False):
    folder, shared = replay_args(scene, variant, zero_velocity)
    if replay_ready(folder):
        print(f"Replay đã có: {folder}")
        return
    suffix = "-velocity-off" if zero_velocity else ""
    with progress.phase(f"render-{scene}-{variant}{suffix}"):
        research("replay", *shared, "--cameras", *REPLAY_CAMERAS, "--fps", REPLAY_FPS,
                 "--scale", REPLAY_SCALE, "--interpolation", REPLAY_INTERPOLATION)

def ground_for(scene):
    return manifest_for(scene).with_name(f"ground-{tag_cameras}.json")

def prepare_ground(scene, raw):
    ground = ground_for(scene)
    if not ground.is_file() or read_json(ground)["manifest_sha256"] != sha256(manifest_for(scene)):
        research("ground", "--raw", raw, "--manifest", manifest_for(scene), "--output", ground)
    return ground

def deform_seed_for(scene):
    # v2 giữ thêm điểm LiDAR của từng khung cho loss LiDAR–mesh của người xa.
    return manifest_for(scene).with_name(f"deform-seed-{tag_cameras}-v2.npz")

def prepare_deform_seed(scene, raw):
    # Điểm LiDAR trong cuboid (cả chuỗi và từng khung) và điểm lấp, tô màu từ các camera, cho người xa ở R3.
    seed = deform_seed_for(scene)
    if not seed.is_file():
        research("deform_seed", "--raw", raw, "--scene", scene, "--cameras", *CAMERAS, "--output", seed)
    return seed

def keypoints_for(scene):
    return manifest_for(scene).with_name(f"keypoints-{tag_cameras}-v1.json")

def prepare_keypoints(scene, raw):
    # Keypoint 2D (ViTPose) trên bbox của mọi quan sát người đi bộ: hông, gối, cổ chân… cho người xa ở R3.
    output = keypoints_for(scene)
    if not output.is_file():
        shell("uv", "pip", "install", "--python", REPO / ".deps/train-env/bin/python", "transformers==4.53.2")
        research("keypoints", "--raw", raw, "--scene", scene, "--manifest", manifest_for(scene), "--output", output)
    return output

def train_scene(scene, variant, raw, quality_set):
    folder = run_folder(scene, variant)
    checkpoint = latest_checkpoint(folder, include_initial=True)
    done = checkpoint_step(checkpoint)
    if done >= steps_for(variant):
        print(f"{scene} {variant} đã hoàn thành {done} bước")
        return
    live = ["--quality-set", quality_set, "--quality-dir", RESULTS / f"{scene}-{variant_dir(variant)}-quality",
            "--quality-every", QUALITY_EVERY]
    if variant in ("R2", "R3"):
        live.extend(["--pose-monitor-dir", RESULTS / f"{scene}-{variant_dir(variant)}-pose",
                     "--pose-monitor-every", QUALITY_EVERY])
    if checkpoint:
        print(f"{scene} {variant} tiếp tục từ {checkpoint.name} sau {done} bước", flush=True)
        research("train", "--resume", folder / "config.yml", "--checkpoint", checkpoint, "--raw", raw,
                 "--steps", steps_for(variant) - done, "--checkpoint-every", CHECKPOINT_EVERY, *live)
        return
    pose = ["--tracks", tracks_for(scene), "--body", BODY] if variant in ("R2", "R3") else []
    if variant == "R3":
        pose.extend(["--ground", prepare_ground(scene, raw), "--deform-seed", prepare_deform_seed(scene, raw),
                     "--keypoints", prepare_keypoints(scene, raw), "--gait-library", GAIT_LIBRARY])
    manifest = ["--manifest", manifest_for(scene)] if variant != "SplatAD" else []
    research("train", "--raw", raw, "--scene", scene, "--variant", variant, *pose, *manifest,
             "--cameras", *CAMERAS, "--output", RUNS, "--run-id", RUN_ID, "--steps", steps_for(variant),
             "--downscales", 0,
             "--checkpoint-every", CHECKPOINT_EVERY,
             "--monitor-every", 0 if variant == "SplatAD" else (200 if STAGE == "smoke" else 500), *live)

def train_with_recovery(scene, variant, raw, quality_set):
    with progress.phase(f"train-{scene}-{variant}"):
        retry_environment(lambda: train_scene(scene, variant, raw, quality_set),
                          lambda: bootstrap("train", repair=True))
        if checkpoint_step(checkpoint_for(scene, variant)) < steps_for(variant):
            raise RuntimeError(f"{scene} {variant} chưa có checkpoint đạt {steps_for(variant)} bước.")

for stage_index, scene in enumerate(SCENES, start=1):
    focus = CURRICULUM.get(scene, {}).get("label", "") if STAGE == "curriculum" else ""
    print(f"[{stage_index}/{len(SCENES)}] Cảnh {scene} · {focus or STAGE}", flush=True)
    if (all(checkpoint_step(checkpoint_for(scene, variant)) >= steps_for(variant) for variant in ("SplatAD", *VARIANTS))
            and all(replay_ready(replay_args(scene, variant)[0]) for variant in ("SplatAD", *VARIANTS))):
        print(f"Cảnh {scene} đã có checkpoint và video, bỏ qua bước train và render.")
        continue
    raw = raw_for(scene)
    quality_set = INPUTS / f"{scene}-{tag_cameras}-rgbq-v{DATA_PROTOCOL_VERSION}.json"
    with progress.phase(f"prepare-{scene}"):
        prepare_scene(scene, raw)
        publish_source(scene, raw, manifest_for(scene))
        if not quality_set.is_file():
            research("quality", "--raw", raw, "--scene", scene, "--cameras", *CAMERAS,
                     "--manifest", manifest_for(scene), "--output", quality_set)
    train_with_recovery(scene, "SplatAD", raw, quality_set)
    render_scene(scene, "SplatAD")  # Video 60 FPS từ checkpoint cuối, như R2 và R3
    for variant in VARIANTS:
        train_with_recovery(scene, variant, raw, quality_set)
        render_scene(scene, variant)

## OmniRe gốc để đối chứng

Cell này train OmniRe của DriveStudio tại commit đã pin với `configs/omnire.yaml` và `pandaset/6cams`. Thiết lập chung với SplatAD, R2 và R3: cùng cảnh, sáu camera ở độ phân giải gốc, toàn bộ 80 frame dùng để train, LiDAR Pandar64, 10.000 bước, seed 42 và cùng file `smpl.pkl` từ bước pose 4D-Humans. Mặt nạ bầu trời lấy từ SegFormer-B5 Cityscapes, đúng checkpoint OmniRe dùng, chạy qua Transformers. Bảng so sánh bốn mô hình được ghi trong `artifacts/<run_id>/<scene>-summary/`. Notebook xuất video ba camera trước từ checkpoint cuối ở 60 FPS cho thẻ **OmniRe** của dashboard. Camera đi theo cùng đường nội suy như các bản render khác. OmniRe biểu diễn thời gian theo chỉ số frame, nên người và xe trong video OmniRe đổi trạng thái ở 10 Hz. Ở mỗi checkpoint, mesh SMPL khởi tạo và mesh OmniRe đã tinh chỉnh được chiếu lên RGB gốc cho cùng các quan sát R2/R3 đang theo dõi.

Vòng train lặp lại `tools/train.py` của DriveStudio và lưu thêm `resume.pth` (mô hình, Adam, bước, trạng thái ngẫu nhiên) mỗi 1.000 bước, nên lượt bị ngắt train tiếp từ đó. Trong lúc train, notebook ghi gradient và loss mỗi 25 bước, đo PSNR/SSIM/L1 và chiếu mesh SMPL lên RGB gốc mỗi 500 bước, giống R2 và R3. OmniRe nạp sẵn ảnh và mặt nạ dạng float32, cần khoảng 30 GB RAM ở độ phân giải gốc. Runtime L4 của Colab đáp ứng được.

In [ ]:
COMPARE_OMNIRE = True
OMNIRE_DATA = Path("/content/omnire-processed")
OMNIRE_CHECKPOINT_EVERY = 2500

def omnire(*args):
    shell("bash", "scripts/omnire-baseline.sh", *args, cwd=REPO)

def run_omnire(scene):
    raw = raw_for(scene)
    output = RUNS / f"{scene}-OmniRe"
    quality_set = INPUTS / f"{scene}-{tag_cameras}-rgbq-v{DATA_PROTOCOL_VERSION}.json"
    shared = ["--scene", scene, "--processed", OMNIRE_DATA]
    with progress.phase(f"omnire-prepare-{scene}"):
        retry_environment(lambda: omnire("prepare", *shared, "--raw", raw, "--pose", LOCAL_POSES),
                          lambda: bootstrap("omnire", repair=True))
    with progress.phase(f"omnire-train-{scene}"):
        # Train tiếp từ resume.pth sau khi mất kết nối; đo chất lượng và chiếu mesh mỗi 500 bước.
        omnire("train", *shared, "--output", output, "--run-id", RUN_ID, "--steps", STEPS,
               "--smpl", SMPL_PICKLE, "--checkpoint-every", OMNIRE_CHECKPOINT_EVERY, "--raw", raw,
               "--quality-set", quality_set, "--result", RESULTS / f"{scene}-OmniRe-quality",
               "--manifest", manifest_for(scene), "--monitor", RESULTS / f"{scene}-OmniRe-pose")
    with progress.phase(f"omnire-render-{scene}"):
        omnire("video", *shared, "--output", output, "--run-id", RUN_ID, "--steps", STEPS, "--raw", raw,
               "--manifest", manifest_for(scene), "--fps", REPLAY_FPS,
               "--result", RESULTS / f"{scene}-OmniRe-final-{REPLAY_FPS}fps-1x-smooth")

if COMPARE_OMNIRE:
    assert tag_cameras == "surround", "OmniRe gốc dùng cấu hình pandaset/6cams."
    with progress.phase("environment-omnire"):
        bootstrap("omnire")
    for scene in SCENES:
        run_omnire(scene)

for scene in SCENES:
    reports = [RESULTS / f"{scene}-{variant_dir(name)}-quality" / "quality.json" for name in ("SplatAD", *VARIANTS, "OmniRe")]
    reports = [path for path in reports if path.is_file()]
    if len(reports) > 1:
        research("summary", "--reports", *reports, "--output", RESULTS / f"{scene}-summary")

## Metric và đối chứng

Cell này bổ sung metric cho các replay đã xuất. Đánh giá dùng ảnh và tia tại mốc cảm biến gốc. Khi có cả R2 và R3, `compare` ghi chênh lệch R3 so với R2 trên cùng quan sát. Đặt `CHECK_VELOCITY=True` để render thêm đối chứng R2/vận tốc 0. Đặt `RUN_GRADIENT_PROBES=True` để chạy probe RGB/LiDAR từ checkpoint sau khi xuất video.

In [ ]:
PROBES_ONLY = STAGE != "confirm"
CHECK_VELOCITY = STAGE in ("smoke", "pilot", "confirm")
RUN_GRADIENT_PROBES = STAGE in ("smoke", "pilot", "confirm")
folders = []
staged = set()
metric_scope = "probes" if PROBES_ONLY else "full"

def stage_for_evaluation(scene):
    if scene not in staged:
        raw_for(scene)
        staged.add(scene)

def run_pending_check(scene, phase, output, module, *args):
    if output.is_file() and progress.records.get(phase, {}).get("status") == "complete":
        return
    stage_for_evaluation(scene)
    with progress.phase(phase):
        research(module, *args)

for scene in SCENES:
    for variant in VARIANTS:
        velocity_modes = (False, True) if CHECK_VELOCITY and variant == "R2" else (False,)
        for zero_velocity in velocity_modes:
            folder, shared = replay_args(scene, variant, zero_velocity)
            if not replay_ready(folder):
                stage_for_evaluation(scene)
                render_scene(scene, variant, zero_velocity=zero_velocity)
            run_pending_check(scene, f"evaluate-{metric_scope}-{folder.name}", folder / "metrics.json",
                              "evaluate", *shared, *(["--probes"] if PROBES_ONLY else []))
            folders.append(folder)
if len(VARIANTS) > 1 or (CHECK_VELOCITY and "R2" in VARIANTS):
    with progress.phase("compare"):
        research("compare", "--runs", *folders, "--output", RESULTS / "comparison")

if RUN_GRADIENT_PROBES:
    for scene in SCENES:
        for variant in VARIANTS:
            if variant in ("R2", "R3"):
                checkpoint = checkpoint_for(scene, variant)
                # Include the trained step in the filename and phase name so different checkpoints can be inspected.
                phase = f"{scene}-{variant}-{checkpoint.stem}-probe"
                output = RESULTS / f"{phase}.json"
                run_pending_check(scene, phase, output, "probe", "--config", run_folder(scene, variant) / "config.yml",
                                  "--checkpoint", checkpoint, "--manifest", manifest_for(scene), "--output", output)

## Dashboard sau khi render

Chọn cảnh rồi chuyển giữa **Gốc** và **Render**. Bấm tên camera để xem riêng và **Cả ba** để trở lại. Trong **Pose SMPL**, nút **Tới khung hình** mở đúng camera và mốc đang so khớp.

In [ ]:
if "dashboard" not in globals() or dashboard.poll() is not None:
    dashboard = subprocess.Popen([sys.executable, "-m", "research.dashboard",
                                  "--artifacts", str(ARTIFACTS), "--port", "8765"], cwd=REPO)
    time.sleep(1)
output.serve_kernel_port_as_iframe(8765, height=1000)